# Aula 5 — IDHM por Unidade da Federação

Nesta aula, vamos abrir a `Tabela4.csv`, limpar os dados, responder a perguntas sobre o IDHM e criar gráficos de linhas com **pandas** e **matplotlib**.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Abrir o CSV

O arquivo tem uma linha de título antes do cabeçalho, separador `;`, vírgula decimal e codificação Windows-1252.

In [ ]:
df = pd.read_csv(
    "Tabela4.csv",
    sep=";",
    decimal=",",
    encoding="cp1252",
    skiprows=1,
)

df.head()

## 2. Limpar colunas vazias

`dropna(axis=1, how="all")` remove somente as colunas totalmente vazias. Também identificamos as colunas que representam anos e garantimos que seus valores sejam numéricos.

In [ ]:
df = df.dropna(axis=1, how="all")
df.columns = df.columns.str.strip()

anos = [c for c in df.columns if str(c).isdigit() and len(str(c)) == 4]
df[anos] = df[anos].apply(pd.to_numeric, errors="coerce")

print("Anos encontrados:", anos)
print("Dimensões da tabela:", df.shape)
df.info()

## 3. Ordenar os estados pelo IDHM de 2024

In [ ]:
ranking_2024 = (
    df[["Sigla", "Estado", "2024"]]
    .sort_values("2024", ascending=False)
    .reset_index(drop=True)
)

ranking_2024.index = ranking_2024.index + 1
ranking_2024.index.name = "Posição"
ranking_2024

## 4. Estado com a maior melhora entre 1991 e 2024

A melhora é calculada pela diferença `IDHM de 2024 - IDHM de 1991`.

In [ ]:
df["Melhora_1991_2024"] = df["2024"] - df["1991"]

maior_melhora = df.loc[
    df["Melhora_1991_2024"].idxmax(),
    ["Sigla", "Estado", "1991", "2024", "Melhora_1991_2024"],
]

print("Estado com a maior melhora:", maior_melhora["Estado"])
print(f"IDHM em 1991: {maior_melhora['1991']:.3f}")
print(f"IDHM em 2024: {maior_melhora['2024']:.3f}")
print(f"Melhora: {maior_melhora['Melhora_1991_2024']:.3f}")

**Resposta:** Tocantins teve a maior melhora, passando de **0,369** em 1991 para **0,797** em 2024, um aumento de **0,428**.

## 5. Algum estado apresentou piora?

Consideramos que houve piora quando o IDHM de 2024 é menor que o de 1991.

In [ ]:
estados_com_piora = df.loc[
    df["Melhora_1991_2024"] < 0,
    ["Sigla", "Estado", "1991", "2024", "Melhora_1991_2024"],
]

if estados_com_piora.empty:
    print("Não. Todos os estados apresentaram melhora entre 1991 e 2024.")
else:
    print("Estados que apresentaram piora:")
    display(estados_com_piora)

## 6. Passar do formato largo para o formato longo (`melt`)

No formato largo, cada ano ocupa uma coluna. No formato longo, cada linha representa uma combinação de estado e ano. A coluna calculada de melhora não participa do `melt`.

In [ ]:
id_vars = ["Sigla", "Código", "Estado"]

df_longo = df.melt(
    id_vars=id_vars,
    value_vars=anos,
    var_name="Ano",
    value_name="IDH",
)

df_longo["Ano"] = df_longo["Ano"].astype(int)
df_longo["IDH"] = pd.to_numeric(df_longo["IDH"], errors="coerce")
df_longo = df_longo.dropna(subset=["IDH"])

df_longo.head()

## 7. Plotar apenas Minas Gerais

In [ ]:
mg = df_longo[df_longo["Sigla"] == "MG"].sort_values("Ano")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mg["Ano"], mg["IDH"], marker="o", linewidth=2.2, color="#1f77b4")
ax.set_title("Evolução do IDHM de Minas Gerais (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDHM")
ax.set_ylim(0.3, 0.9)
ax.set_xticks(mg["Ano"])
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

## 8. Plotar a evolução do IDHM de cada estado

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for sigla, grupo in df_longo.groupby("Sigla"):
    grupo = grupo.sort_values("Ano")
    ax.plot(
        grupo["Ano"],
        grupo["IDH"],
        marker="o",
        markersize=3,
        linewidth=1.5,
        label=sigla,
    )

ax.set_title("Evolução do IDHM por estado (1991–2024)")
ax.set_xlabel("Ano")
ax.set_ylabel("IDHM")
ax.set_ylim(0.3, 0.9)
ax.legend(ncol=3, bbox_to_anchor=(1.02, 1), loc="upper left", title="UF")
fig.tight_layout()
plt.show()

## Conclusões

- O Distrito Federal tem o maior IDHM em 2024: **0,866**.
- Tocantins teve a maior melhora entre 1991 e 2024: **0,428**.
- Nenhuma Unidade da Federação terminou 2024 com IDHM inferior ao de 1991.